# Lab 7 Student Version (Google Colab)
## Build a Custom CNN for CIFAR-10

This notebook is a **student-learning version** of the attached Lab 7 guide. It includes **every topic and step** (Tasks 1–4 + comparison + lab review) and adds explanations of **what you are learning** and **why each step matters**.

## Lab overview
In this lab, you will build and train a **custom convolutional neural network (CNN)** from scratch on the **CIFAR-10** dataset. You’ll also learn how to **inspect intermediate feature maps** from a convolution layer using two methods:
- **Forward hooks** (capture outputs during a full forward pass)
- **Forward slicing** (manually run part of the forward pass)

### In this lab, you will
- Define a CNN using `nn.Module`
- Train the CNN on CIFAR-10
- Perform and verify a forward pass
- Visualize feature maps using hooks and forward slicing

### Estimated completion time
- **35 minutes**

### Runtime type
- Set runtime type to: **CPU**


# Task 0 (Colab setup): Imports and environment check

### What you are learning
You verify that your environment has the required libraries. This reduces setup errors and makes your lab reproducible.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Lab runtime recommendation: CPU")


# Task 1: Defining `CNNNet(nn.Module)`

### What you are learning
You are learning how to define a CNN architecture using `nn.Module`. A CNN uses:
- **Convolution layers** to detect spatial patterns (edges, textures, shapes),
- **ReLU** for nonlinearity, and
- **MaxPool** to reduce spatial size while keeping important features.

### Steps (from the lab)
1. Import necessary libraries.
2. Define a custom CNN model (`CNNNet`).
3. Instantiate the model and print it.
4. Count total trainable parameters and print the result.


In [ ]:
# 1. Import necessary libraries (already imported in Task 0).
import torch
import torch.nn as nn
import torch.nn.functional as F

# 2. Define a custom CNN model.
class CNNNet(nn.Module):
    def __init__(self):
        super(CNNNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=0)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 6 * 6, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = x.view(-1, 32 * 6 * 6)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# 3. Instantiate model and print it.
model = CNNNet()
print(model)


In [ ]:
# 4. Count total trainable parameters.
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Total Trainable Parameters: {total_params}")


### Task 1 explanation (learning takeaways)
- `Conv2d(in_channels, out_channels, kernel_size)` learns filters that detect patterns in images.
- `MaxPool2d(2,2)` down-samples (shrinks) the feature maps to make computation cheaper and focus on the strongest activations.
- `view(-1, 32*6*6)` flattens feature maps into a vector so fully-connected layers can process them.
- Counting parameters helps you understand model capacity (more parameters → more expressive but may overfit).


# Task 2: Performing a forward pass with Conv + Pool + ReLU

### What you are learning
You are learning how to:
- load CIFAR-10 with `torchvision`,
- get a small batch from a `DataLoader`,
- run a forward pass through your CNN,
- verify the input/output tensor shapes so you know your architecture works.

### Steps (from the lab)
1. Import torchvision and transforms.
2. Load CIFAR-10 dataset and create a DataLoader with `batch_size=4`.
3. Instantiate the model and get one batch.
4. Forward pass.
5. Print and verify input/output shapes.


In [ ]:
# 1. Import torchvision and transforms (already imported in Task 0).
import torchvision
import torchvision.transforms as transforms

# 2. Load CIFAR-10 dataset.
transform = transforms.ToTensor()
trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=4, shuffle=True)

# 3. Instantiate model and get one batch.
model = CNNNet()
dataiter = iter(trainloader)
images, labels = next(dataiter)

# 4. Forward pass.
outputs = model(images)

# 5. Check output shape.
print(f"Input shape:  {images.shape}")
print(f"Output shape: {outputs.shape}")


### Task 2 explanation (learning takeaways)
- CIFAR-10 images are **3×32×32** (3 color channels).
- A batch of 4 images has shape `(4, 3, 32, 32)`.
- The model output has shape `(batch_size, 10)` because CIFAR-10 has **10 classes**.
- Shape checks are a fast way to catch architecture mistakes early.


# Task 3: Training the CNN for 1–2 epochs

### What you are learning
You are learning the standard training loop for image classification:
1) forward pass → 2) compute loss → 3) backward pass → 4) optimizer step  
You will also track a simple training accuracy estimate while training.

### Steps (from the lab)
1. Import optimizer library.
2. Set up training: loss, optimizer, device, and move model to device.
3. Train for `epochs = 2` and print loss + accuracy each epoch.


In [ ]:
# 1. Import the required library.
import torch.optim as optim

# 2. Set up training.
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# The lab shows a device check; the lab runtime recommends CPU, but this keeps the code portable.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Using device:", device)


In [ ]:
# 3. Training loop.
epochs = 2
for epoch in range(epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(trainloader):.4f} - Accuracy: {correct/total:.4f}")


### Task 3 explanation (learning takeaways)
- `CrossEntropyLoss` is the standard loss for multi-class classification.
- `Adam` is a popular optimizer that adapts learning rates per parameter.
- Moving data/model to `device` keeps the code compatible with CPU or GPU environments.
- Accuracy here is computed on training batches and is just a quick training signal (not a full evaluation).


# Task 4: Visualizing intermediate feature maps (hooks and slicing)

### What you are learning
You are learning how to inspect what a CNN is “seeing” inside. The output of a convolution layer is a set of **feature maps** (one per learned filter). Visualizing them helps you understand:
- what patterns are being activated,
- how early layers detect edges/textures,
- and how to debug CNN behavior.

This task shows two techniques:
- **Option A:** forward hooks (capture outputs during the full forward pass)
- **Option B:** forward slicing (manually run to a specific layer)

---

## Option A: Using forward hooks on `conv1`
### Steps (from the lab)
1. Create a dictionary to store feature maps + hook function.
1.1 Register hook on `model.conv1`.
1.2 Run inference on a single image.
1.3 Visualize 16 feature maps.
1.4 Remove the hook to avoid memory leaks.


In [ ]:
# Option A: using forward hooks on conv1.
feature_maps = {}

def get_activation(name):
    def hook(model, input, output):
        feature_maps[name] = output.detach()
    return hook

# 1.1 Register hook.
hook_handle = model.conv1.register_forward_hook(get_activation('conv1'))

# 1.2 Inference on a single image.
img, _ = trainset[0]                 # img is a tensor because we used ToTensor()
img_batch = img.unsqueeze(0).to(device)

model.eval()
_ = model(img_batch)                 # run full forward pass to trigger hook

# 1.3 Visualize feature maps from conv1 (hook).
fmaps = feature_maps['conv1'].squeeze().cpu()  # shape: (16, H, W)

plt.figure(figsize=(12, 10))
for i in range(16):
    plt.subplot(4, 4, i+1)
    plt.imshow(fmaps[i], cmap='viridis')
    plt.title(f'Hook Map {i}', fontsize=8)
    plt.axis('off')

plt.suptitle("Feature Maps from conv1 using Hook", fontsize=14)
plt.tight_layout()
plt.show()

# 1.4 Remove hook to avoid memory leaks.
hook_handle.remove()


---

## Option B: Forward slicing
### Steps (from the lab)
2.1 Use the same image as before.
2.2 Manually slice the forward pass up to `conv1` (conv1 → ReLU).
2.3 Visualize 16 feature maps.


In [ ]:
# Option B: forward slicing.
model.eval()

# 2.1 Use same image.
img, _ = trainset[0]
img_batch = img.unsqueeze(0).to(device)

# 2.2 Manually slice forward pass up to conv1.
with torch.no_grad():
    x = model.conv1(img_batch)
    x = torch.relu(x)

# 2.3 Visualize sliced output.
x = x.squeeze().cpu()

plt.figure(figsize=(12, 10))
for i in range(16):
    plt.subplot(4, 4, i+1)
    plt.imshow(x[i], cmap='plasma')
    plt.title(f'Slice Map {i}', fontsize=8)
    plt.axis('off')

plt.suptitle("Feature Maps from conv1 using Forward Slicing", fontsize=14)
plt.tight_layout()
plt.show()


### Task 4 explanation (learning takeaways)
- **Forward hooks** capture intermediate outputs automatically during a normal full forward pass.
- **Forward slicing** runs only part of the model manually to reach a layer you want to inspect.
- Both methods help you debug and understand how CNNs interpret image features.


# Comparison: hooks vs. forward slicing

| Aspect | Forward hook | Forward slicing |
|---|---|---|
| Setup | Attach a hook to any layer with `register_forward_hook()` | Manually run selected layers in code |
| What it does | Captures intermediate outputs during a normal forward pass | Computes only the portion you explicitly run |
| Best for | Debugging/inspecting models without changing `forward()` | Quick experiments when the layer path is simple |
| Control | High (you can access inputs/outputs at runtime) | Moderate (only tensors you compute are available) |
| Flexibility | Works well even in nested/complex models | Easier when the model is small or sequential |
| Complexity | Moderate (you must manage hooks and remove them) | Simple (just run the layers you need) |


# Lab review

1. Forward hooks allow you to extract intermediate layer outputs without modifying the model’s forward method.  
A. True  
B. False  

2. What is the main reason for using `MaxPool2d` in a convolutional neural network?  
A. To normalize the features  
B. To increase the number of channels  
C. To reduce spatial dimensions and retain important features  
D. To increase training time  

---

## STOP
You have successfully completed this lab.


<details>
<summary><strong>Optional self-check answers (click to expand)</strong></summary>

1. **A (True)**  
2. **C** — pooling reduces spatial size while keeping important activations.

</details>
